In [1]:
import numpy as np
from scipy.linalg import pinvh
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
base = Path.home() / "Desktop" / "Dissertation" / "kappa-kinisi"
d = np.load(base / "data" / "kinisi_rw_data_1.npz")

In [3]:
cov = d["cov"]

msd = d["msd"]

timestep = msd[0, :, 0]

no = msd[0, :, 3]

dim = 3

D_true = 1.0

In [4]:
A = np.column_stack([timestep, np.ones_like(timestep)])

n = no.shape[0]

ana = np.zeros((n, n))

for i in range(n):
    for j in range(i, n):
        ana[i, j] = 8 * dim**2 * timestep[i]**2 / (dim * no[j])
        ana[j, i] = ana[i, j]
        
true_lmin = np.linalg.eigvalsh(ana)[0]

In [5]:
def r_none(m):
    return m


In [6]:
def r_mineig(m, kappa=1e16):
    v, V = np.linalg.eigh(m)
    fl = v[-1] / kappa
    return (V * np.where(v < fl, fl, v)) @ V.T

In [7]:
def r_mineig_k3(m):
    return r_mineig(m, 1e3)

In [8]:
def r_ridge(m, kappa=1e3):
    v = np.linalg.eigvalsh(m)

    delta = (v[-1] - kappa * v[0]) / (kappa - 1)
    
    return m + delta * np.eye(m.shape[0]) if delta > 0 else m

In [9]:
def r_adaptive(m, c=0.25):
    v, V = np.linalg.eigh(m)

    if v[0] < 0:
        fl = c * (-v[0])
        v = np.where(v < fl, fl, v)
        
    return (V * v) @ V.T

In [10]:
def r_clip_mp(m):
    v, V = np.linalg.eigh(m)

    q = m.shape[0] / max(no)

    sigma2 = np.median(v[v > 0])

    mp_max = sigma2 * (1 + np.sqrt(q)) ** 2

    v = np.where(v < mp_max, mp_max, v)
    
    return (V * v) @ V.T

In [11]:
def r_nls_analytical(m):
    v, V = np.linalg.eigh(m)

    p = len(v)

    nobs = max(no)

    lam = v.copy()
    
    lam_pos = np.maximum(lam, 1e-12)

    h = nobs ** (-1 / 3)

    f = np.zeros(p)

    for i in range(p):
        x = (lam_pos[i] - lam_pos) / (h * lam_pos + 1e-30)

        k = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)

        f[i] = np.mean(k / (h * lam_pos + 1e-30))

    d_shrunk = lam_pos / ((np.pi * (p / nobs) * lam_pos * f) ** 2 + 0.25)
    
    d_shrunk = np.maximum(d_shrunk, 1e-10)


    return (V * d_shrunk) @ V.T

In [12]:
def r_taper(m, c=0.5):
    v, V = np.linalg.eigh(m)

    scale = v[-1]

    soft = 0.5 * (v + np.sqrt(v**2 + (c * scale * 1e-3) ** 2))

    soft = np.maximum(soft, scale * 1e-12)
    
    return (V * soft) @ V.T

In [13]:
reconditioners = {"none": r_none,
                  "mineig 1e16": r_mineig,
                  "mineig 1e3": r_mineig_k3,
                  "ridge": r_ridge,
                  "adaptive": r_adaptive,
                  "clip MP": r_clip_mp,
                  "NLS analytic": r_nls_analytical,
                  "taper": r_taper}

In [14]:
def i_plain(m):
    return np.linalg.inv(m)


def i_pinvh(m):
    return pinvh(m)


def i_pinvh_rcond(m, rcond=1e-6):
    return pinvh(m, rtol=rcond)


def i_tsvd(m, k=None):
    v, V = np.linalg.eigh(m)
    if k is None:
        k = np.sum(v > v[-1] * 1e-6)
    keep = np.zeros_like(v)
    idx = np.argsort(v)[::-1][:k]
    keep[idx] = 1.0 / v[idx]
    keep[v <= 0] = 0.0
    return (V * keep) @ V.T


def i_tikhonov(m, lam=None):
    if lam is None:
        lam = np.trace(m) / m.shape[0] * 1e-3
    return np.linalg.inv(m + lam * np.eye(m.shape[0]))


inverses = {"plain": i_plain,
            "pinvh": i_pinvh,
            "pinvh rcond": i_pinvh_rcond,
            "tsvd": i_tsvd,
            "tikhonov": i_tikhonov}

In [15]:
def gls_D(sigma_inv, y):
    lhs = A.T @ sigma_inv @ A
    rhs = A.T @ sigma_inv @ y
    try:
        beta = np.linalg.solve(lhs, rhs)
        return beta[0] / (2 * dim)
    except np.linalg.LinAlgError:
        return np.nan

In [16]:
n_sim = 500

results = {}

for rname, rfn in reconditioners.items():
    m0 = rfn(cov[0])

    ev = np.linalg.eigvalsh(m0)

    lmin = ev[0]

    kappa = ev[-1] / lmin if lmin > 0 else np.inf

    lmin_err = abs(lmin - true_lmin)




    for iname, ifn in inverses.items():
        Ds = []

        for k in range(n_sim):
            m = rfn(cov[k])

            try:
                Ds.append(gls_D(ifn(m), msd[k, :, 1]))

            except (np.linalg.LinAlgError, ValueError):
                Ds.append(np.nan)

        Ds = np.array(Ds)

        good = Ds[np.isfinite(Ds)]

        good = good[(good > -5) & (good < 5)]
        
        results[(rname, iname)] = {"lmin_err": lmin_err,
                                    "log_kappa": np.log10(kappa) if np.isfinite(kappa) else 16.0,
                                    "D_rmse": np.sqrt(np.mean((good - D_true) ** 2)) if len(good) else np.nan,
                                    "D_bias": np.median(good) - D_true if len(good) else np.nan}



In [17]:
print(f"{'reconditioning x inverse':28s} {'lmin_err':>9s} {'logK':>6s} {'D_rmse':>9s} {'D_bias':>8s}")


for (r, i), v in sorted(results.items(), key=lambda x: (np.nan_to_num(x[1]['D_rmse'], nan=1e9))):
    print(f"{r+' x '+i:28s} {v['lmin_err']:9.3f} {v['log_kappa']:6.2f} "
          f"{v['D_rmse']:9.4f} {v['D_bias']:+8.4f}")

reconditioning x inverse      lmin_err   logK    D_rmse   D_bias
none x pinvh                    33.567  16.00    0.0146  -0.0024
none x plain                    33.567  16.00    0.0146  -0.0024
NLS analytic x pinvh             0.003  15.55    0.0146  -0.0024
mineig 1e16 x pinvh              0.003  16.01    0.0146  -0.0024
none x tikhonov                 33.567  16.00    0.0225  -0.0035
none x pinvh rcond              33.567  16.00    0.0236  -0.0032
NLS analytic x tsvd              0.003  15.55    0.0237  -0.0043
NLS analytic x pinvh rcond       0.003  15.55    0.0237  -0.0043
none x tsvd                     33.567  16.00    0.0237  -0.0043
mineig 1e16 x tsvd               0.003  16.01    0.0237  -0.0043
mineig 1e16 x pinvh rcond        0.003  16.01    0.0237  -0.0043
adaptive x plain                 8.388   4.02    0.0306  -0.0071
adaptive x pinvh                 8.388   4.02    0.0306  -0.0071
adaptive x pinvh rcond           8.388   4.02    0.0317  -0.0078
adaptive x tsvd          